In [1]:
# Cell 1: Fix NumPy version and install required packages
!pip uninstall numpy -y -q
!pip install numpy==1.24.3 --quiet
!pip install scipy==1.10.1 --quiet
!pip install ultralytics --upgrade --quiet

# Verify installations
import sys
import subprocess

# Check versions
print("Checking package versions...")
!python -c "import numpy; print(f'NumPy: {numpy.__version__}')"
!python -c "import scipy; print(f'SciPy: {scipy.__version__}')"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.24.3 which is incompatible.
Checking package versions...
NumPy: 2.2.6
SciPy: 1.15.3


In [3]:
#
import sys
import subprocess
import importlib

print("=" * 60)
print("FIXING NUMPY VERSION CONFLICT")
print("=" * 60)

# First, let's see what's installed
print("\nCurrent NumPy version:")
!python -c "import numpy; print(f'NumPy: {numpy.__version__}')" 2>/dev/null || echo "NumPy not imported"

print("\nUninstalling problematic packages...")
!pip uninstall numpy scipy matplotlib -y -q 2>/dev/null

print("\nInstalling compatible versions...")
!pip install "numpy<2" --quiet
!pip install "scipy<1.11" --quiet
!pip install "matplotlib<3.7" --quiet

print("\nVerifying installations:")
!python -c "import numpy; print(f'✓ NumPy: {numpy.__version__}')"
!python -c "import scipy; print(f'✓ SciPy: {scipy.__version__}')"

print("\n" + "=" * 60)
print("IMPORTANT: You MUST RESTART THE KERNEL now!")
print("1. Go to 'Kernel' menu")
print("2. Select 'Restart Kernel'")
print("3. Then run the next cell")
print("=" * 60)

FIXING NUMPY VERSION CONFLICT

Current NumPy version:
NumPy: 1.21.5

Uninstalling problematic packages...

Installing compatible versions...

Verifying installations:
✓ NumPy: 1.21.5
✓ SciPy: 1.8.0

IMPORTANT: You MUST RESTART THE KERNEL now!
1. Go to 'Kernel' menu
2. Select 'Restart Kernel'
3. Then run the next cell


In [6]:
#
print("AFTER KERNEL RESTART - Checking GPU...")
print("=" * 60)

# Check if we have the right numpy
try:
    import numpy as np
    print(f"✓ NumPy version: {np.__version__}")
    if np.__version__.startswith('2'):
        print("⚠️  WARNING: Still NumPy 2.x! Run Cell 1 again.")
except:
    print("✗ NumPy not loaded")

# Check PyTorch and CUDA
try:
    import torch
    print(f"✓ PyTorch version: {torch.__version__}")
    print(f"✓ CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU Memory: {gpu_memory:.2f} GB")
    else:
        print("✗ No GPU detected. Training will use CPU.")
        
except Exception as e:
    print(f"✗ Error: {e}")

AFTER KERNEL RESTART - Checking GPU...
✓ NumPy version: 1.21.5
✓ PyTorch version: 2.7.0
✓ CUDA available: True
✓ GPU: NVIDIA RTX 5000 Ada Generation
✓ GPU Memory: 33.81 GB


In [1]:
# Nano
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Nano/nano26'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    # List directory to help debug
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 model...")
model = YOLO('yolo26n-seg.pt')  # Use nano model for testing

# SIMPLE training with minimal options to avoid conflicts
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,  # Start small
        imgsz=640,
        batch=8,
        device=device,  # This is the key - GPU or CPU
        project=output_dir,
        name='yolo26_test',
        exist_ok=True,
        verbose=True,
        workers=2,  # Keep low to avoid issues
        val=True,
        save=True
    )
    
    print("\n✓ Training completed successfully!")
    
except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")
    
    # Fallback to CPU
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=8,  # Smaller for CPU
        device='cpu',  # Force CPU
        project=output_dir,
        name='yolo26_cpu',
        exist_ok=True,
        verbose=True
    )

SIMPLE YOLO26 GPU TRAINING
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 model...

Starting training...
Ultralytics 8.4.10 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, i

In [2]:
# Nano_sahi
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 segmentation model...")
model = YOLO('yolo26n-seg.pt')  # nano segmentation model

# SIMPLE training with small-object oriented settings
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,
        imgsz=960,          # a bit larger to help small objects
        batch=8,
        device=device,
        project=output_dir,
        name='yolo26_sahi_ready',
        exist_ok=True,
        verbose=True,
        workers=2,
        val=True,
        save=True
    )
    print("\n✓ Training completed successfully!")

except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=960,
        batch=4,
        device='cpu',
        project=output_dir,
        name='yolo26_sahi_ready_cpu',
        exist_ok=True,
        verbose=True
    )
    
    print(f"✓ Saved SAHI result for {img_path.name} into {output_dir}")

print("\n✓ Completed SAHI sliced inference on all images.")



SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 segmentation model...

Starting training...
Ultralytics 8.4.10 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False,

In [3]:
#small
# Cell 3: SIMPLE GPU TRAINING CODE
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Small'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    # List directory to help debug
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 model...")
model = YOLO('yolo26s-seg.pt')  # Use nano model for testing

# SIMPLE training with minimal options to avoid conflicts
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,  # Start small
        imgsz=640,
        batch=8,
        device=device,  # This is the key - GPU or CPU
        project=output_dir,
        name='yolo26_test',
        exist_ok=True,
        verbose=True,
        workers=2,  # Keep low to avoid issues
        val=True,
        save=True
    )
    
    print("\n✓ Training completed successfully!")
    
except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")
    
    # Fallback to CPU
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=8,  # Smaller for CPU
        device='cpu',  # Force CPU
        project=output_dir,
        name='yolo26_cpu',
        exist_ok=True,
        verbose=True
    )

SIMPLE YOLO26 GPU TRAINING
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 model...

Starting training...
New https://pypi.org/project/ultralytics/8.4.8 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.7 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=tor

In [4]:
#small_sahi
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Small'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 segmentation model...")
model = YOLO('yolo26s-seg.pt')  # nano segmentation model

# SIMPLE training with small-object oriented settings
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,
        imgsz=960,          # a bit larger to help small objects
        batch=8,
        device=device,
        project=output_dir,
        name='yolo26_sahi_ready',
        exist_ok=True,
        verbose=True,
        workers=2,
        val=True,
        save=True
    )
    print("\n✓ Training completed successfully!")

except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=960,
        batch=4,
        device='cpu',
        project=output_dir,
        name='yolo26_sahi_ready_cpu',
        exist_ok=True,
        verbose=True
    )
    
    print(f"✓ Saved SAHI result for {img_path.name} into {output_dir}")

print("\n✓ Completed SAHI sliced inference on all images.")



SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 segmentation model...

Starting training...
Ultralytics 8.4.8 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, 

In [7]:
#medium
# Cell 3: SIMPLE GPU TRAINING CODE
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Medium'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    # List directory to help debug
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 model...")
model = YOLO('yolo26m-seg.pt')  # Use nano model for testing

# SIMPLE training with minimal options to avoid conflicts
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,  # Start small
        imgsz=640,
        batch=8,
        device=device,  # This is the key - GPU or CPU
        project=output_dir,
        name='yolo26_test',
        exist_ok=True,
        verbose=True,
        workers=2,  # Keep low to avoid issues
        val=True,
        save=True
    )
    
    print("\n✓ Training completed successfully!")
    
except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")
    
    # Fallback to CPU
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=8,  # Smaller for CPU
        device='cpu',  # Force CPU
        project=output_dir,
        name='yolo26_cpu',
        exist_ok=True,
        verbose=True
    )

SIMPLE YOLO26 GPU TRAINING
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 model...

Starting training...
Ultralytics 8.4.8 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, im

In [7]:
# medium_sahi
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Medium'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 segmentation model...")
model = YOLO('yolo26m-seg.pt')  # nano segmentation model

# SIMPLE training with small-object oriented settings
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,
        imgsz=960,          # a bit larger to help small objects
        batch=8,
        device=device,
        project=output_dir,
        name='yolo26_sahi_ready',
        exist_ok=True,
        verbose=True,
        workers=2,
        val=True,
        save=True
    )
    print("\n✓ Training completed successfully!")

except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=960,
        batch=4,
        device='cpu',
        project=output_dir,
        name='yolo26_sahi_ready_cpu',
        exist_ok=True,
        verbose=True
    )
    
    print(f"✓ Saved SAHI result for {img_path.name} into {output_dir}")

print("\n✓ Completed SAHI sliced inference on all images.")



SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 segmentation model...

Starting training...
Ultralytics 8.4.8 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, 

In [9]:
#Large
# Cell 3: SIMPLE GPU TRAINING CODE
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/large'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    # List directory to help debug
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 model...")
model = YOLO('yolo26l-seg.pt')  # Use nano model for testing

# SIMPLE training with minimal options to avoid conflicts
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,  # Start small
        imgsz=640,
        batch=8,
        device=device,  # This is the key - GPU or CPU
        project=output_dir,
        name='yolo26_test',
        exist_ok=True,
        verbose=True,
        workers=2,  # Keep low to avoid issues
        val=True,
        save=True
    )
    
    print("\n✓ Training completed successfully!")
    
except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")
    
    # Fallback to CPU
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=8,  # Smaller for CPU
        device='cpu',  # Force CPU
        project=output_dir,
        name='yolo26_cpu',
        exist_ok=True,
        verbose=True
    )

SIMPLE YOLO26 GPU TRAINING
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 model...

Starting training...
Ultralytics 8.4.8 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, im

In [4]:
#large_sahi
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/large'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 segmentation model...")
model = YOLO('yolo26l-seg.pt')  # nano segmentation model

# SIMPLE training with small-object oriented settings
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,
        imgsz=960,          # a bit larger to help small objects
        batch=8,
        device=device,
        project=output_dir,
        name='yolo26_sahi_ready',
        exist_ok=True,
        verbose=True,
        workers=2,
        val=True,
        save=True
    )
    print("\n✓ Training completed successfully!")

except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=960,
        batch=4,
        device='cpu',
        project=output_dir,
        name='yolo26_sahi_ready_cpu',
        exist_ok=True,
        verbose=True
    )
    
    print(f"✓ Saved SAHI result for {img_path.name} into {output_dir}")

print("\n✓ Completed SAHI sliced inference on all images.")



SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 segmentation model...

Starting training...
Ultralytics 8.4.8 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, 

In [5]:
#XTRALARGE
# Cell 3: SIMPLE GPU TRAINING CODE
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/xtraLarge'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    # List directory to help debug
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 model...")
model = YOLO('yolo26x-seg.pt')  # Use nano model for testing

# SIMPLE training with minimal options to avoid conflicts
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,  # Start small
        imgsz=640,
        batch=8,
        device=device,  # This is the key - GPU or CPU
        project=output_dir,
        name='yolo26_test',
        exist_ok=True,
        verbose=True,
        workers=2,  # Keep low to avoid issues
        val=True,
        save=True
    )
    
    print("\n✓ Training completed successfully!")
    
except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")
    
    # Fallback to CPU
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=8,  # Smaller for CPU
        device='cpu',  # Force CPU
        project=output_dir,
        name='yolo26_cpu',
        exist_ok=True,
        verbose=True
    )

SIMPLE YOLO26 GPU TRAINING
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 model...

Starting training...
Ultralytics 8.4.8 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, im

In [6]:
#XTRALARGE_sahi
import os
import torch
from ultralytics import YOLO

print("=" * 60)
print("SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS")
print("=" * 60)

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ Cleared GPU cache")

# Set paths
dataset_path = '/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/xtraLarge'
os.makedirs(output_dir, exist_ok=True)

# Check if dataset exists
if not os.path.exists(dataset_path):
    print(f"✗ Dataset not found: {dataset_path}")
    data_dir = os.path.dirname(dataset_path)
    if os.path.exists(data_dir):
        print(f"\nFiles in {data_dir}:")
        for f in os.listdir(data_dir)[:10]:
            print(f"  - {f}")
else:
    print(f"✓ Dataset found: {dataset_path}")

# Set device
if torch.cuda.is_available():
    device = 0  # Use first GPU
    print(f"✓ Using GPU: {device}")
else:
    device = 'cpu'
    print("⚠️  Using CPU (no GPU available)")

# Load model (will auto-download)
print("\nLoading YOLO26 segmentation model...")
model = YOLO('yolo26x-seg.pt')  # nano segmentation model

# SIMPLE training with small-object oriented settings
print("\nStarting training...")
try:
    results = model.train(
        data=dataset_path,
        epochs=200,
        imgsz=960,          # a bit larger to help small objects
        batch=8,
        device=device,
        project=output_dir,
        name='yolo26_sahi_ready',
        exist_ok=True,
        verbose=True,
        workers=2,
        val=True,
        save=True
    )
    print("\n✓ Training completed successfully!")

except Exception as e:
    print(f"\n✗ Training error: {str(e)}")
    print("\nTrying CPU instead...")

    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=960,
        batch=4,
        device='cpu',
        project=output_dir,
        name='yolo26_sahi_ready_cpu',
        exist_ok=True,
        verbose=True
    )
    
    print(f"✓ Saved SAHI result for {img_path.name} into {output_dir}")

print("\n✓ Completed SAHI sliced inference on all images.")



SIMPLE YOLO26 GPU TRAINING WITH SMALL-OBJECT FOCUS
✓ Cleared GPU cache
✓ Dataset found: /home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml
✓ Using GPU: 0

Loading YOLO26 segmentation model...

Starting training...
Ultralytics 8.4.8 🚀 Python-3.10.12 torch-2.7.0 CUDA:0 (NVIDIA RTX 5000 Ada Generation, 32240MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ranjan/Documents/yolo26/earlystageapple/GreenApples.v4-yolo26smallobjectsegmentation.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, 

In [8]:
import os
import torch
from ultralytics import YOLO
from pathlib import Path
import cv2

# Simple version using Ultralytics plotting
model_path = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Nano/yolo26_sahi_ready/weights/best.pt'
input_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Images/RGB'
output_dir = '/home/ranjan/Documents/yolo26/earlystageapple/Experiment/Images/output_SAHIsimples'

os.makedirs(output_dir, exist_ok=True)

# Load model
model = YOLO(model_path)

# Get all images
image_files = []
for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
    image_files.extend(Path(input_dir).glob(f'*{ext}'))
    image_files.extend(Path(input_dir).glob(f'*{ext.upper()}'))

print(f"Found {len(image_files)} images")

# Process each image
for img_path in image_files:
    # Run inference
    results = model(str(img_path), save=False, verbose=False)
    
    # Save visualized result
    for result in results:
        # Save with Ultralytics plotting (includes masks and boxes)
        plotted = result.plot()  # This creates the visualization
        
        # Save to output directory
        output_path = os.path.join(output_dir, f"result_{img_path.name}")
        cv2.imwrite(output_path, plotted)
        
        # Also save the raw masks if needed
        if result.masks is not None:
            # Create mask-only image
            h, w = plotted.shape[:2]
            mask_img = np.zeros((h, w, 3), dtype=np.uint8)
            
            masks = result.masks.data.cpu().numpy() if result.masks.data.is_cuda else result.masks.data.numpy()
            for mask in masks:
                mask_resized = cv2.resize(mask, (w, h))
                mask_binary = (mask_resized > 0.5).astype(np.uint8)
                mask_img[mask_binary == 1] = (0, 255, 0)  # Green masks
            
            mask_path = os.path.join(output_dir, f"mask_{img_path.name}")
            cv2.imwrite(mask_path, mask_img)

print(f"Results saved to: {output_dir}")

Found 55 images
Results saved to: /home/ranjan/Documents/yolo26/earlystageapple/Experiment/Images/output_SAHIsimples
